In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("pyyaml")
install("scikit-learn")
install("tqdm")
install("matplotlib")
install("pandas")
install("git+https://github.com/openai/CLIP.git")

import clip
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

REPO_URL = "https://github.com/dvydinh/ralm_industrial_anomaly_detection.git"
REPO_DIR = "/content/ralm_industrial_anomaly_detection"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, REPO_DIR)

from raml.models.raml_model import RAMLModel
from raml.losses.combined_loss import MACCLLoss
from raml.data.mvtec_dataset import MVTecDataset
from raml.utils.metrics import compute_per_category_metrics



In [ ]:
import os
import subprocess
MVTEC_ROOT = "/content/drive/MyDrive/ralm/data/mvtec_anomaly_detection"
SAVE_DIR = "/content/drive/MyDrive/ralm"
for d in ["models", "metrics", "plots"]:
    os.makedirs(os.path.join(SAVE_DIR, d), exist_ok=True)
if not os.path.exists(os.path.join(MVTEC_ROOT, "bottle")):
    import kagglehub
    import shutil
    path = kagglehub.dataset_download("ipythonx/mvtec-ad")
    os.makedirs(MVTEC_ROOT, exist_ok=True)
    shutil.copytree(path, MVTEC_ROOT, dirs_exist_ok=True)
else:
    print(f"Dataset already exists at {MVTEC_ROOT}")


In [ ]:
CONFIG = {
    "clip_model": "ViT-B/16",
    "feature_dim": 512,
    "hidden_dim": 256,
    "num_heads": 4,
    "dropout": 0.1,

    "margin_base": 0.5,
    "lambda_sigma": 0.0,
    "lambda_resolution": 0.0,
    "original_resolution": 900,
    "model_resolution": 224,
    "temperature": 0.07,
    "alpha": 1.0,
    "beta": 1.0,
    "gamma": 0.5,

    "epochs": 25,
    "batch_size": 64,
    "lr": 5e-5,
    "weight_decay": 0.02,
    "max_grad_norm": 1.0,
    
    "encode_chunk_size": 512,
    "num_workers": 12,
    "persistent_workers": True,
    "pin_memory": True,

    "visual_weight": 0.85,
    "text_weight": 0.15,

    "use_test_anomalies": True,
    "train_ratio": 0.8,
    "seed": 42,

    "difficulty_tracker": {
        "momentum": 0.7,
        "weight_scale": 0.5,
        "clip_min": 0.5,
        "clip_max": 2.0,
    },
}



In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model, preprocess = clip.load(CONFIG["clip_model"], device=device)
tokenizer = clip.tokenize

model = RAMLModel(
    clip_model,
    tokenizer,
    feature_dim=CONFIG["feature_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    num_heads=CONFIG["num_heads"],
    dropout=CONFIG["dropout"],
    visual_weight=CONFIG["visual_weight"],
    text_weight=CONFIG["text_weight"],
).to(device)

model.extractor.encode_chunk_size = CONFIG["encode_chunk_size"]

loss_fn = MACCLLoss(
    feature_dim=CONFIG["hidden_dim"],
    margin_base=CONFIG["margin_base"],
    lambda_sigma=CONFIG["lambda_sigma"],
    lambda_resolution=CONFIG["lambda_resolution"],
    original_resolution=CONFIG["original_resolution"],
    model_resolution=CONFIG["model_resolution"],
    temperature=CONFIG["temperature"],
    alpha=CONFIG["alpha"],
    beta=CONFIG["beta"],
    gamma=CONFIG["gamma"],
    difficulty_cfg=CONFIG["difficulty_tracker"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)

train_ds = MVTecDataset(
    MVTEC_ROOT,
    categories=[d for d in os.listdir(MVTEC_ROOT) if os.path.isdir(os.path.join(MVTEC_ROOT, d))],
    split="train",
    transform=preprocess,
    use_test_anomalies=CONFIG["use_test_anomalies"],
    train_ratio=CONFIG["train_ratio"],
    seed=CONFIG["seed"],
)
test_ds = MVTecDataset(
    MVTEC_ROOT,
    categories=[d for d in os.listdir(MVTEC_ROOT) if os.path.isdir(os.path.join(MVTEC_ROOT, d))],
    split="test",
    transform=preprocess,
    use_test_anomalies=CONFIG["use_test_anomalies"],
    train_ratio=CONFIG["train_ratio"],
    seed=CONFIG["seed"],
)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True, 
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"], 
    persistent_workers=CONFIG["persistent_workers"]
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG["batch_size"], shuffle=False, 
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"], 
    persistent_workers=CONFIG["persistent_workers"]
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total")
print(f"Train: {len(train_ds)} samples, Test: {len(test_ds)} samples")



In [ ]:
import time
from tqdm.auto import tqdm

def train_one_epoch(model, loader, loss_fn, optimizer, device, max_grad_norm):
    model.train()
    total_loss = 0.0
    n = 0
    for images, labels, cats in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        out = model(images, categories=list(cats))

        loss_maccl, info = loss_fn(out["features"], labels, list(cats))
        loss_bce = F.binary_cross_entropy_with_logits(out["logits"], labels)
        loss = loss_maccl + loss_bce

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


def evaluate(model, loader, device):
    model.eval()
    all_labels, all_scores, all_cats = [], [], []
    with torch.no_grad():
        for images, labels, cats in tqdm(loader, desc="Eval", leave=False):
            images = images.to(device)
            out = model(images, categories=list(cats))
            all_scores.extend(out["scores"].cpu().tolist())
            all_labels.extend(labels.tolist())
            all_cats.extend(list(cats))
    return compute_per_category_metrics(all_cats, all_labels, all_scores)

print(f"Starting training for {CONFIG['epochs']} epochs")
print("=" * 60)

best_auroc = 0.0
history = []
best_details = {}
model_save_path = os.path.join(SAVE_DIR, "models", "fixed_margin.pt")
latest_path = os.path.join(SAVE_DIR, "models", "latest_fixed_margin.pt")
start_epoch = 1

if os.path.exists(latest_path):
    print(f"Resuming from {latest_path}")
    checkpoint = torch.load(latest_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint.get("epoch", 0) + 1
    best_auroc = checkpoint.get("best_auroc", 0.0)
    history = checkpoint.get("history", [])
    best_details = checkpoint.get("best_details", {})
    print(f"Resumed at epoch {start_epoch-1} with best AUROC {best_auroc*100:.2f}%")

for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    t0 = time.time()
    avg_loss = train_one_epoch(
        model, train_loader, loss_fn, optimizer, device, CONFIG["max_grad_norm"]
    )
    cat_metrics, macro_metrics = evaluate(model, test_loader, device)
    mean_auroc = macro_metrics['auroc']
    elapsed = time.time() - t0

    history.append({"epoch": epoch, "loss": avg_loss, "auroc": mean_auroc})

    print(f"Epoch {epoch:2d}/{CONFIG['epochs']}  loss={avg_loss:.4f}  "
          f"AUROC={mean_auroc*100:.2f}%  time={elapsed:.0f}s")


    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": CONFIG,
        "best_auroc": best_auroc,
        "history": history,
        "best_details": best_details
    }, latest_path)
    
    if mean_auroc > best_auroc:
        best_auroc = mean_auroc
        best_details = cat_metrics
        best_macro = macro_metrics
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "auroc": mean_auroc,
        }, model_save_path)
        print(f"  -> New best! Saved to {model_save_path}")

print("=" * 60)
print(f"Best AUROC: {best_auroc*100:.2f}%")




In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

print("\nPer-category results (best epoch):")
print("-" * 40)
for cat, score in sorted(best_details.items()):
    print(f"  {cat:15s}: {score*100:.2f}%")
print("-" * 40)
print(f"  {'MEAN':15s}: {best_auroc*100:.2f}%")

# Save JSON
result = {
    "method": "fixed_margin",
    "config": CONFIG,
    "mean_auroc": round(best_auroc * 100, 2),
    "per_category": {k: round(v * 100, 2) for k, v in best_details.items()},
    "training_history": history,
}
json_path = os.path.join(SAVE_DIR, "metrics", "result_fixed_margin.json")
with open(json_path, "w") as f:
    json.dump(result, f, indent=2)

# Save CSV
csv_path = os.path.join(SAVE_DIR, "metrics", "result_fixed_margin.csv")
df = pd.DataFrame(list(best_details.items()), columns=["Category", "AUROC"])
df["AUROC"] = df["AUROC"] * 100
df.loc[len(df)] = ["MEAN", best_auroc * 100]
df.to_csv(csv_path, index=False)

# Plot Loss & AUROC
epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]
aurocs = [h["auroc"] * 100 for h in history]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(epochs, losses, "r-", label="Loss")
ax2.plot(epochs, aurocs, "b-", label="AUROC")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="r")
ax2.set_ylabel("AUROC (%)", color="b")
plt.title("Training Curve - fixed_margin")
plot_path = os.path.join(SAVE_DIR, "plots", "curve_fixed_margin.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"All outputs strictly saved to {SAVE_DIR}")

